In [1]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT_DIR))

import random
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI

from src import evaluation as ev
from src.judge import evaluate_answer
from src.prompt import ANSWER_MODEL, INSTRUCTIONS, PROMPT_TEMPLATE
from src.rag_helper import RAGBase
from src.search import vector_search

load_dotenv(ROOT_DIR / ".env")
client = OpenAI()
helper = RAGBase(index=None, llm_client=None, instructions=INSTRUCTIONS,
                 prompt_template=PROMPT_TEMPLATE, model=ANSWER_MODEL)

records = pd.read_csv(ROOT_DIR / "data" / "rag_answer.csv").to_dict(orient="records")

In [17]:
def judge_record(rec):
    context = helper.build_context(vector_search(rec["question"]))
    result = evaluate_answer(rec["question"], rec["llm_answer"], context, client=client)
    return {"question": rec["question"], "context": context,
            "llm_answer": rec["llm_answer"], **result}

with ThreadPoolExecutor(max_workers=6) as pool:
    results = ev.map_progress(pool, records, judge_record)

df_g = pd.DataFrame(results)
df_g.to_csv(ROOT_DIR / "data" / "groundedness_eval.csv", index=False)

print(df_g["groundedness"].value_counts(normalize=True).round(3))
print(f"Judge cost: ${df_g['cost'].sum():.2f}")

  0%|          | 0/355 [00:00<?, ?it/s]

groundedness
GROUNDED              0.932
PARTIALLY_GROUNDED    0.068
Name: proportion, dtype: float64
Judge cost: $0.49


In [22]:
FAKE_CLAIMS = [
    "Additionally, this failure is always preceded by a vibration spike above 12 mm/s.",
    "Per OEM guidance, the spindle bearing must be replaced every 500 operating hours.",
    "This failure mode causes roughly 40% of all unplanned downtime in CNC plants.",
]

random.seed(42)
sample = random.sample(records, 30)
injected = [
    {**rec, "llm_answer": rec["llm_answer"] + "\n\n" + random.choice(FAKE_CLAIMS)}
    for rec in sample
]
injected_claims = [next(c for c in FAKE_CLAIMS if inj["llm_answer"].endswith(c)) for inj in injected]

with ThreadPoolExecutor(max_workers=6) as pool:
    inj_results = ev.map_progress(pool, injected, judge_record)

df_inj = pd.DataFrame(inj_results)
caught = (df_inj["groundedness"] != "GROUNDED").sum()
print(f"Injected hallucinations caught: {caught}/{len(df_inj)}")


random.seed(42)
sample = random.sample(records, 30)
injected_claims = [random.choice(FAKE_CLAIMS) for _ in sample]

KEYWORDS = {FAKE_CLAIMS[0]: "vibration", FAKE_CLAIMS[1]: "500", FAKE_CLAIMS[2]: "40%"}

df_inj["injected_claim"] = injected_claims
df_inj["claim_named"] = df_inj.apply(
    lambda r: any(KEYWORDS[r["injected_claim"]] in c.lower() for c in r["unsupported_claims"]),
    axis=1,
)

baseline = df_g.set_index("question")["groundedness"]
df_inj["baseline"] = df_inj["question"].map(baseline)
clean = df_inj[df_inj["baseline"] == "GROUNDED"]

print(f"Injected claim explicitly named:      {df_inj['claim_named'].sum()}/30")
print(f"Caught where original answer was clean: "
      f"{(clean['groundedness'] != 'GROUNDED').sum()}/{len(clean)}")

  0%|          | 0/30 [00:00<?, ?it/s]

Injected hallucinations caught: 30/30
Injected claim explicitly named:      30/30
Caught where original answer was clean: 29/29


In [23]:
import re

def perturb(answer):
    m = re.search(r"\b\d{2,4}\b", answer)          # first number with 2-4 digits
    if not m:
        return None
    wrong = str(int(m.group()) * 3 // 2 + 1)        # e.g. 200 -> 301
    return answer[:m.start()] + wrong + answer[m.end():]

random.seed(7)
candidates = [r for r in records if baseline.get(r["question"]) == "GROUNDED"
              and perturb(r["llm_answer"])]
subtle = [{**r, "llm_answer": perturb(r["llm_answer"])} for r in random.sample(candidates, 30)]

with ThreadPoolExecutor(max_workers=6) as pool:
    sub_results = ev.map_progress(pool, subtle, judge_record)

df_sub = pd.DataFrame(sub_results)
print(f"Subtle number changes caught: {(df_sub['groundedness'] != 'GROUNDED').sum()}/30")

  0%|          | 0/30 [00:00<?, ?it/s]

Subtle number changes caught: 29/30


In [24]:
spot = df_g.sample(20, random_state=1).reset_index(drop=True)

for i, row in spot.iterrows():
    print(f"=================== #{i + 1}  Judge says: {row['groundedness']}")
    print("QUESTION:", row["question"])
    print("\nCONTEXT:\n", row["context"])
    print("\nANSWER:\n", row["llm_answer"])
    print("\nUNSUPPORTED CLAIMS (judge):", row["unsupported_claims"])
    print()

=================== #1  Judge says: GROUNDED
QUESTION: What kind of voltage, current, or speed pattern usually shows up right before a power failure?

CONTEXT:
 Power Failure
Q: What is the sensor signature of Power Failure?
A: - Voltage, current, power, torque, or rotational speed deviates from the normal load-dependent pattern.
- Commanded speed differs persistently from measured speed.
- Current or power rises while speed falls, suggesting overload or mechanical resistance.
- Sudden voltage loss or fluctuation coincides with the failure event.

Power Failure
Q: How exactly is machine power calculated from torque and speed?
A: `Power [W] = Torque [Nm] × Rotational speed [rad/s]`, and since rotational speed is usually recorded in rpm, that's `Torque [Nm] × Rotational speed [rpm] × (2π / 60)`. Power Failure triggers when this computed value falls below 3500 W (underpowered) or above 9000 W (overpowered) — monitor it as its own derived metric rather than reading torque and speed separat

In [27]:
my_verdicts = [
    "agree", "agree", "agree", "agree", "agree",
    "agree", "agree", "agree", "disagree", "agree",
    "agree", "disagree","agree", "agree", "disagree",
    "agree","agree", "disagree","agree","disagree"
    
]  

assert len(my_verdicts) == 20
print(f"Agreement with manual review: {my_verdicts.count('agree')}/20")

Agreement with manual review: 15/20
